In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import wandb
from lightning.pytorch.loggers import WandbLogger

from torchmetrics.detection.mean_ap import MeanAveragePrecision

YOLOV5_DIR = Path('/work/external/yolov5')
if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))
from utils.general import non_max_suppression

PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_ROOT))

from classification_pipeline.caries_model import LitToothClassifier, build_classification_resize_pipeline
# (Ide importáld be a LitYOLOv5 osztályt is abból a fájlból, ahová esetleg kiszervezted, 
# vagy másold be a notebookba, ha még nem szervezted ki!)
# from detection_pipeline.yolo_model import LitYOLOv5 

ModuleNotFoundError: No module named 'utils'

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

yolo_ckpt_path = '/work/output/checkpoints/yolov5-finetune-best.ckpt' 
classifier_ckpt_path = '/work/output/checkpoints/classification/full_test-v1.ckpt'

wandb.init(
    project="tooth-e2e-evaluation",
    name="yolo-resnet-pipeline-eval",
    job_type="evaluation"
)

print("Loading YOLOv5...")
yolo_model = LitYOLOv5.load_from_checkpoint(yolo_ckpt_path)
yolo_model.to(DEVICE)
yolo_model.eval()

print("Loading ResNet Classifier...")
classifier_model = LitToothClassifier.load_from_checkpoint(classifier_ckpt_path, num_classes=2)
classifier_model.to(DEVICE)
classifier_model.eval()

resize_transform = build_classification_resize_pipeline(224)

map_metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True)

print("Running End-to-End Evaluation...")

metric_preds = []
metric_targets = []

with torch.no_grad():
    for batch in tqdm(e2e_test_loader, desc="E2E Progress"):
        images, gt_targets = batch 
        images = images.to(DEVICE)
        
        yolo_raw_output = yolo_model(images)
        if isinstance(yolo_raw_output, (tuple, list)):
            yolo_preds = yolo_raw_output[0]
        else:
            yolo_preds = yolo_raw_output
            
        nms_preds = non_max_suppression(yolo_preds, conf_thres=0.5, iou_thres=0.45, max_det=32)
        
        for i, det in enumerate(nms_preds):
            pred_boxes = []
            pred_scores = []
            pred_labels = []
            
            if det is not None and len(det) > 0:
                boxes = det[:, :4] # xyxy
                
                for box in boxes:
                    x_min, y_min, x_max, y_max = box.int().tolist()
                    
                    x_min, y_min = max(0, x_min), max(0, y_min)
                    
                    img_np = images[i].cpu().permute(1, 2, 0).numpy() * 255.0
                    img_np = img_np.astype(np.uint8)
                    tooth_crop = img_np[y_min:y_max, x_min:x_max]
                    
                    if tooth_crop.size == 0:
                        continue
                        
                    input_tensor = resize_transform(image=tooth_crop)['image'].unsqueeze(0).to(DEVICE)
                    
                    logits = classifier_model(input_tensor)
                    probs = torch.softmax(logits, dim=1)[0]
                    
                    caries_prob = probs[1].item()
                    
                    pred_class = 1 if caries_prob > 0.5 else 0
                    
                    final_score = caries_prob if pred_class == 1 else probs[0].item()
                    
                    pred_boxes.append([x_min, y_min, x_max, y_max])
                    pred_scores.append(final_score)
                    pred_labels.append(pred_class)
            
            metric_preds.append({
                "boxes": torch.tensor(pred_boxes, dtype=torch.float32, device=DEVICE).reshape(-1, 4),
                "scores": torch.tensor(pred_scores, dtype=torch.float32, device=DEVICE),
                "labels": torch.tensor(pred_labels, dtype=torch.long, device=DEVICE)
            })

            metric_targets.append({
                "boxes": gt_targets[i]["boxes"].to(DEVICE),
                "labels": gt_targets[i]["caries_labels"].to(DEVICE)
            })

print("Computing End-to-End mAP...")
map_metric.update(metric_preds, metric_targets)
results = map_metric.compute()

print(f"E2E mAP @ IoU 50:95: {results['map']:.4f}")
print(f"E2E mAP @ IoU 50:    {results['map_50']:.4f}")
print(f"E2E mAR @ MaxDets:   {results['mar_100']:.4f}")

wandb.log({
    "e2e/map_50_95": results['map'].item(),
    "e2e/map_50": results['map_50'].item(),
    "e2e/map_75": results['map_75'].item(),
    "e2e/mar_100": results['mar_100'].item(),
})

wandb.finish()
print("E2E Evaluation finished! Check W&B.")